<a href="https://colab.research.google.com/github/irezendegarcia-prog/python_sis_26_3_periodo/blob/Italo_Rezende/trabalho_bimestre/italo_rezende_trabalho_bimestral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabalho Bimestral — Projeto Final: Análise Exploratória de Dados (EDA)

**Aluno:** Italo Rezende  
**Base escolhida:** Diamonds (`italo_rezende_diamonds.csv`) — 53.940 registros reais  
**Fonte:** Conjunto público de dados *Diamonds* (disponível no Kaggle e na biblioteca Seaborn / `mwaskom/seaborn-data`)

## 1. Contexto

A base de dados **Diamonds** reúne **53.940 registros reais** de diamantes comercializados, contendo **10 atributos** que descrevem suas características físicas, métricas de classificação (os *4Cs*: quilate, corte, cor e pureza) e o preço de venda em dólares americanos (`price`).

Principais colunas da base:
- **`carat`**: peso do diamante em quilates;
- **`cut`**: qualidade do corte (*Fair*, *Good*, *Very Good*, *Premium*, *Ideal*);
- **`color`**: classificação de cor, de *D* (melhor) até *J* (pior);
- **`clarity`**: grau de pureza (*I1*, *SI2*, *SI1*, *VS2*, *VS1*, *VVS2*, *VVS1*, *IF*);
- **`depth`** e **`table`**: proporções percentuais da lapidação;
- **`price`**: preço em dólares (USD);
- **`x`, `y`, `z`**: dimensões físicas em milímetros (comprimento, largura e profundidade).

O objetivo desta análise exploratória é inspecionar a estrutura da base, extrair estatísticas descritivas e investigar padrões e curiosidades na formação do preço dos diamantes.

In [ ]:
# 1 - Contexto: Importações (PEP 8), carregamento da base e inspeção (.info())
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

arquivo_csv = "italo_rezende_diamonds.csv"

# Carrega o CSV local se existir; caso contrário, baixa a base real via seaborn e salva o CSV
if os.path.exists(arquivo_csv):
    df = pd.read_csv(arquivo_csv)
    print(f"Base carregada localmente de '{arquivo_csv}'.")
else:
    df = sns.load_dataset("diamonds")
    df.to_csv(arquivo_csv, index=False, encoding="utf-8")
    print(f"Base real 'diamonds' carregada via seaborn e salva em '{arquivo_csv}'.")

print(f"Dimensões da base: {df.shape[0]} linhas x {df.shape[1]} colunas\n")

print("=== Primeiras 5 linhas (head) ===")
print(df.head())

print("\n=== Estrutura e tipos de dados (.info()) ===")
df.info()

print("\n=== Verificação de valores ausentes (NaN) ===")
print(df.isna().sum())

# Remove registros com dimensões físicas zeradas (inconsistências conhecidas na base)
df_limpo = df[(df["x"] > 0) & (df["y"] > 0) & (df["z"] > 0)].copy()
print(f"\nRegistros válidos após remover dimensões zeradas: {len(df_limpo)}")


## 2. Estatísticas Descritivas

Nesta etapa, utilizamos funções do Pandas para calcular médias, medianas, desvios-padrão, contagens por categoria, valores mínimos e máximos, além de uma visão agrupada por qualidade de corte (`cut`).

In [ ]:
# 2 - Estatísticas descritivas com Pandas
print("=== Resumo estatístico geral (.describe()) ===")
print(df_limpo.describe().round(2))

print("\n=== Indicadores principais de Preço (USD) e Quilates (carat) ===")
print(f"Preço médio:   US$ {df_limpo['price'].mean():,.2f}")
print(f"Preço mediano: US$ {df_limpo['price'].median():,.2f}")
print(f"Preço mínimo:  US$ {df_limpo['price'].min():,.2f}")
print(f"Preço máximo:  US$ {df_limpo['price'].max():,.2f}")
print(f"Quilate médio: {df_limpo['carat'].mean():.2f} ct (mín: {df_limpo['carat'].min():.2f} | máx: {df_limpo['carat'].max():.2f})")

print("\n=== Contagem de diamantes por qualidade de corte (value_counts) ===")
print(df_limpo["cut"].value_counts())

ordem_corte = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
resumo_corte = (
    df_limpo.groupby("cut", observed=False)
    .agg(
        quantidade=("price", "count"),
        preco_medio=("price", "mean"),
        preco_mediano=("price", "median"),
        preco_min=("price", "min"),
        preco_max=("price", "max"),
        quilate_medio=("carat", "mean"),
    )
    .reindex(ordem_corte)
    .round(2)
)

print("\n=== Estatísticas de preço e quilate por qualidade de corte ===")
print(resumo_corte)


## 3. Visualização de Dados

Os dois gráficos abaixo revelam uma curiosidade clássica da base **Diamonds**:
1. **Gráfico 1 (Barras):** mostra que diamantes de corte **Ideal** (a melhor nota de lapidação) possuem, surpreendentemente, um **preço médio menor** do que diamantes de corte **Premium** e **Fair**.
2. **Gráfico 2 (Dispersão):** explica esse padrão ao mostrar a relação fortemente crescente entre **peso em quilates (`carat`)** e **preço (`price`)** — diamantes maiores tendem a ter cortes menos perfeitos, puxando a média de preço dessas categorias para cima.

In [ ]:
# 3 - Visualizações customizadas (título, eixos, cores e rótulos)
sns.set_theme(style="whitegrid")

# ----------------------------------------------------------
# Gráfico 1: Preço Médio por Qualidade de Corte (Barras)
# ----------------------------------------------------------
cores_corte = ["#8C564B", "#E69F00", "#56B4E9", "#0072B2", "#009E73"]

fig, ax = plt.subplots(figsize=(9, 5))
barras = ax.bar(
    resumo_corte.index,
    resumo_corte["preco_medio"],
    color=cores_corte,
    width=0.6,
    edgecolor="#222222",
    linewidth=0.8,
)

ax.set_title(
    "Curiosidade: Preço Médio dos Diamantes por Qualidade de Corte",
    fontsize=13,
    fontweight="bold",
    pad=14,
)
ax.set_xlabel("Qualidade do Corte (cut)", fontsize=11, labelpad=8)
ax.set_ylabel("Preço Médio (USD)", fontsize=11, labelpad=8)
ax.set_ylim(0, resumo_corte["preco_medio"].max() * 1.18)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for barra, quilate in zip(barras, resumo_corte["quilate_medio"]):
    altura = barra.get_height()
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        altura + 90,
        f"US$ {altura:,.0f}\n({quilate:.2f} ct)",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        color="#222222",
    )

plt.tight_layout()
plt.show()

# ----------------------------------------------------------
# Gráfico 2: Relação Quilates x Preço por Corte (Dispersão)
# ----------------------------------------------------------
# Amostra estratificada/aleatória (até 5.000 pontos) para manter o gráfico legível
amostra = df_limpo.sample(n=min(5000, len(df_limpo)), random_state=42)

paleta_dispersao = {
    "Fair": "#8C564B",
    "Good": "#E69F00",
    "Very Good": "#56B4E9",
    "Premium": "#0072B2",
    "Ideal": "#009E73",
}

fig, ax = plt.subplots(figsize=(9.5, 5.5))
for corte in ordem_corte:
    sub = amostra[amostra["cut"] == corte]
    ax.scatter(
        sub["carat"],
        sub["price"],
        label=corte,
        color=paleta_dispersao[corte],
        alpha=0.45,
        s=22,
        edgecolors="none",
    )

ax.set_title(
    "Relação entre Peso em Quilates e Preço dos Diamantes por Corte",
    fontsize=13,
    fontweight="bold",
    pad=14,
)
ax.set_xlabel("Peso em Quilates (carat)", fontsize=11, labelpad=8)
ax.set_ylabel("Preço (USD)", fontsize=11, labelpad=8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(title="Qualidade do Corte", frameon=True)

plt.tight_layout()
plt.show()
